In [2]:
import pandas as pd
import numpy as np
import re

## Trích xuất các món ăn từ menu

In [20]:
menu_path = 'demo_menu.xlsx'
all_dishes = []

# Đọc tất cả các sheet trong file
xl = pd.ExcelFile(menu_path)

for sheet_name in xl.sheet_names:

    # Bỏ qua header công ty
    menu_raw = pd.read_excel(menu_path, sheet_name=sheet_name, skiprows=4, header=None)
    
    # Dòng chứa ngày là dòng index 1, từ cột index 2 trở đi
    dates = menu_raw.iloc[1, 2:].values

    # Xử lý thân bảng
    menu_content = menu_raw.iloc[2:, 1:].copy()
    # Fill giá trị của cột món ăn
    menu_content.iloc[:,0] = menu_content.iloc[:,0].ffill()
    
    for i, date in enumerate(dates):
        if pd.isna(date): continue # Bỏ qua cột trống nếu có
        
        # Lấy dữ liệu của từng ngày (cột tương ứng)
        temp_menu = menu_content.iloc[:, [0, i+1]].copy()
        temp_menu.columns = ['Loại món', 'Tên món']
        temp_menu['Ngày'] = date
        all_dishes.append(temp_menu)

all_dishes[:6]

[     Loại món Tên món       Ngày
 2   Món chính     NaN 2025-12-29
 3     Món phụ     NaN 2025-12-29
 4         Xào     NaN 2025-12-29
 5        Canh     NaN 2025-12-29
 6    Món chay     NaN 2025-12-29
 7    Món chay     NaN 2025-12-29
 8    Món soup     NaN 2025-12-29
 9   Món chính     NaN 2025-12-29
 10    Món phụ     NaN 2025-12-29
 11        Xào     NaN 2025-12-29
 12       Canh     NaN 2025-12-29
 13       Canh     NaN 2025-12-29
 14       Canh     NaN 2025-12-29,
      Loại món Tên món       Ngày
 2   Món chính     NaN 2025-12-30
 3     Món phụ     NaN 2025-12-30
 4         Xào     NaN 2025-12-30
 5        Canh     NaN 2025-12-30
 6    Món chay     NaN 2025-12-30
 7    Món chay     NaN 2025-12-30
 8    Món soup     NaN 2025-12-30
 9   Món chính     NaN 2025-12-30
 10    Món phụ     NaN 2025-12-30
 11        Xào     NaN 2025-12-30
 12       Canh     NaN 2025-12-30
 13       Canh     NaN 2025-12-30
 14       Canh     NaN 2025-12-30,
      Loại món Tên món       Ngày
 2   Món chí

In [21]:
menu_df = pd.concat(all_dishes, ignore_index=True)
# Làm sạch dữ liệu
menu_df = menu_df.dropna(subset=['Tên món'])
menu_df['Tên món'] = menu_df['Tên món'].str.strip()

# Chuyển cột Ngay sang định dạng datetime để sắp xếp
menu_df['Ngày'] = pd.to_datetime(menu_df['Ngày'], dayfirst=True)

# Tạo cột Next Date để báo dữ liệu này bị trùng
next_dates = menu_df.groupby('Tên món')['Ngày'].shift(-1)
menu_df['Ngày trùng'] = ("trùng với ngày " + next_dates.dt.strftime('%d/%m')).where(next_dates.notna(), np.nan)
menu_df

,Loại món,Tên món,Ngày,Ngày trùng
52,Món chính,Bò nấu gừng,2026-01-02,NaN
53,Món phụ,Đậu nướng tỏi,2026-01-02,NaN
54,Xào,Món sốt lá chanh,2026-01-02,trùng với ngày 20/01
55,Canh,Thịt kho sả ớt,2026-01-02,trùng với ngày 28/01
56,Món chay,Rau rim gừng,2026-01-02,trùng với ngày 06/01
...,...,...,...,...
659,Món chính,Canh sốt nước mắm,2026-02-28,NaN
660,Món phụ,Món kho mỡ hành,2026-02-28,NaN
661,Xào,Cá luộc gừng,2026-02-28,NaN
662,Canh,Gà rim mỡ hành,2026-02-28,NaN


In [22]:
# Tạo menu dict để map với file inventory
menu_dict = {}
for date_val, group in menu_df.groupby('Ngày'):
    # Chuyển ngày về định dạng chuỗi giống s_name (DD-MM-YYYY) để khớp nhau
    date_str = date_val.strftime('%d-%m-%Y')
    menu_dict[date_str] = group[['Loại món', 'Tên món', 'Ngày trùng']].reset_index(drop=True)
menu_dict

{'02-01-2026':      Loại món             Tên món            Ngày trùng
 0   Món chính         Bò nấu gừng                   NaN
 1     Món phụ       Đậu nướng tỏi                   NaN
 2         Xào    Món sốt lá chanh  trùng với ngày 20/01
 3        Canh      Thịt kho sả ớt  trùng với ngày 28/01
 4    Món chay        Rau rim gừng  trùng với ngày 06/01
 5    Món chay   Thịt xào thập cẩm  trùng với ngày 05/01
 6    Món soup      Rau chiên gừng  trùng với ngày 29/01
 7   Món chính    Thịt kho mỡ hành  trùng với ngày 07/01
 8     Món phụ       Đậu chiên tỏi  trùng với ngày 20/01
 9         Xào      Cá kho mỡ hành  trùng với ngày 08/01
 10       Canh  Món nướng lá chanh  trùng với ngày 09/01,
 '03-01-2026':      Loại món               Tên món            Ngày trùng
 0   Món chính       Gà sốt nước mắm  trùng với ngày 07/01
 1     Món phụ         Rau chiên tỏi                   NaN
 2         Xào           Gà hấp gừng  trùng với ngày 05/01
 3        Canh      Đậu chiên ngũ vị  trùng với ngà

## Làm việc với file inventory

In [23]:
# 1. Đọc file (đọc thô toàn bộ để tìm vị trí trước)
inventory_path = 'demo_inventory.xlsx'
inv_raw = pd.read_excel(inventory_path, header=None)
inv_raw

,0,1,2,3,4,5,6,7,8
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,"GLOBO-FEAST CATERING SOLUTIONS CO., LTD",NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,"Address: No. 123 Nowhere Lane, Void District, ...",NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,Tel: +84 (0) 123 456 789,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
3101,28.0,2.0,56,Cá sông rim chua ngọt (Mẫu 129),kg,NaN,0.9,NaN,NaN
3102,28.0,2.0,57,Mì căn luộc chua ngọt (Mẫu 135),kg,NaN,2.0,NaN,NaN
3103,28.0,2.0,58,NaN,NaN,NaN,NaN,NaN,NaN
3104,28.0,2.0,59,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
results_storage = {}

# 1. Tìm vị trí các ngày trong file inventory
day_indices = []
for idx, row in inv_raw.iterrows():
    cell_val = str(row[2]) 
    if re.search(r'Ngày\s+\d+\s+tháng\s+\d+\s+năm\s+\d+', cell_val):
        day_indices.append(idx)
day_indices.append(len(inv_raw))

print(f"Tìm thấy {len(day_indices)-1} ngày trong file Kho.")

Tìm thấy 44 ngày trong file Kho.


In [25]:
# 2. Vòng lặp xử lý và combine với menu
for i in range(len(day_indices) - 1):
    start_pos = day_indices[i]
    end_pos = day_indices[i+1]
    
    # Lấy tên ngày (Dùng làm Key khớp với menu_dict)
    raw_date_str = str(inv_raw.iloc[start_pos, 2])
    match = re.search(r'(\d+)\s+tháng\s+(\d+)\s+năm\s+(\d+)', raw_date_str)
    if match:
        # Tạo s_name đúng định dạng DD-MM-YYYY (VD: 10-02-2026)
        s_name = f"{match.group(1).zfill(2)}-{match.group(2).zfill(2)}-{match.group(3)}"

    # Xử lý dữ liệu xuất kho
    inv_daily = inv_raw.iloc[start_pos:end_pos, 2:].copy()
    
    try:
        # Dòng thứ 3 sau ngày tháng là dòng dữ liệu
        inv_daily.columns = inv_daily.iloc[2]
        inv_daily = inv_daily.iloc[3:].copy()
        inv_daily = inv_daily.dropna(axis=1, how='all')
        inv_daily = inv_daily.reset_index(drop=True)
        
        # Kết hợp với menu dict tạo từ trước đó
        if s_name in menu_dict:
            df_menu = menu_dict[s_name] # Lấy món ăn đã chuẩn bị từ trước
            
            # Tạo khoảng trống giữa 2 bảng (1 cột trắng)
            spacer = pd.DataFrame({' ': [''] * max(len(inv_daily), len(df_menu))})
            
            # Ghép ngang: Kho + Trống + Menu
            # Dùng reset_index để đảm bảo các dòng khớp nhau từ trên xuống
            combined = pd.concat([
                inv_daily.reset_index(drop=True), 
                spacer.reset_index(drop=True), 
                df_menu.reset_index(drop=True)
            ], axis=1)
            status = "Menu founded"
        else:
            combined = inv_daily
            status = "No menu founded"

        # Lưu kết quả đã combine vào kho tạm
        results_storage[s_name] = combined
        print(f"{s_name}: {status}")

    except Exception as e:
        print(f"Eror occur while doing {s_name}: {e}")

02-01-2026: Menu founded
03-01-2026: Menu founded
05-01-2026: Menu founded
06-01-2026: Menu founded
07-01-2026: Menu founded
08-01-2026: Menu founded
09-01-2026: Menu founded
10-01-2026: Menu founded
12-01-2026: Menu founded
13-01-2026: Menu founded
14-01-2026: Menu founded
15-01-2026: Menu founded
16-01-2026: Menu founded
17-01-2026: Menu founded
19-01-2026: Menu founded
20-01-2026: Menu founded
21-01-2026: Menu founded
22-01-2026: Menu founded
23-01-2026: Menu founded
24-01-2026: Menu founded
26-01-2026: Menu founded
27-01-2026: Menu founded
28-01-2026: Menu founded
29-01-2026: Menu founded
30-01-2026: Menu founded
31-01-2026: Menu founded
02-02-2026: Menu founded
03-02-2026: Menu founded
04-02-2026: Menu founded
05-02-2026: Menu founded
06-02-2026: Menu founded
07-02-2026: Menu founded
09-02-2026: Menu founded
10-02-2026: Menu founded
11-02-2026: Menu founded
12-02-2026: Menu founded
13-02-2026: Menu founded
14-02-2026: Menu founded
23-02-2026: Menu founded
24-02-2026: Menu founded


In [26]:
# Tên file xuất ra
final_filename = 'Inventory_Menu_Integration.xlsx'

# Bắt đầu ghi từ bộ nhớ ra file
with pd.ExcelWriter(final_filename, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    for s_name, data in results_storage.items():
        # Ghi dữ liệu vào sheet
        data.to_excel(writer, index=False, sheet_name=s_name)
        print(f'Data: {s_name} has been exported successfully')

print(f"Done: {final_filename}")

Data: 02-01-2026 has been exported successfully
Data: 03-01-2026 has been exported successfully
Data: 05-01-2026 has been exported successfully
Data: 06-01-2026 has been exported successfully
Data: 07-01-2026 has been exported successfully
Data: 08-01-2026 has been exported successfully
Data: 09-01-2026 has been exported successfully
Data: 10-01-2026 has been exported successfully
Data: 12-01-2026 has been exported successfully
Data: 13-01-2026 has been exported successfully
Data: 14-01-2026 has been exported successfully
Data: 15-01-2026 has been exported successfully
Data: 16-01-2026 has been exported successfully
Data: 17-01-2026 has been exported successfully
Data: 19-01-2026 has been exported successfully
Data: 20-01-2026 has been exported successfully
Data: 21-01-2026 has been exported successfully
Data: 22-01-2026 has been exported successfully
Data: 23-01-2026 has been exported successfully
Data: 24-01-2026 has been exported successfully
Data: 26-01-2026 has been exported succe